# NELLWICK AI Film Studio — NELLWICK AI Film Studio — Google Colab worker

This notebook is an interactive batch worker, not an always-on API server. Start it when you want to process queued shots. It checkpoints each uploaded shot to Google Drive and reports job progress to your Studio.

**Before running:** create a worker token in Studio → Settings; connect Google Drive in Studio → Settings; enable a GPU runtime; replace the GitHub repository placeholder in the run cell. Never save your token in the notebook or repository.

The first adapter runs Wan 2.1 T2V 1.3B for short text-to-video shots. Other models are listed as future/benchmark adapters and are not falsely shown as ready. First model load downloads several GB and may take time.


## Google Colab setup

For **Kaggle**, turn Internet on and select GPU in Notebook Settings. Kaggle documents free P100 availability, but queues and usage limits can vary. For **Colab**, select Runtime → Change runtime type → GPU. Colab does not guarantee free GPU availability. Do not use this notebook as a media server, tunnelled backend, or deepfake service.

In [ ]:
# Installs Python dependencies without replacing the CUDA-enabled PyTorch already provided by Kaggle/Colab.
%pip install -q -U "diffusers>=0.35,<0.40" "transformers>=4.51,<5" accelerate safetensors sentencepiece imageio-ffmpeg requests


In [ ]:
import os, getpass, subprocess, sys
NELLWICK_STUDIO_URL = input("Studio URL (for local testing use your deployed Vercel URL): ").strip().rstrip("/")
NELLWICK_WORKER_TOKEN = getpass.getpass("Paste the one-time worker token created in Studio Settings: ").strip()
NELLWICK_MAX_JOBS = input("Maximum jobs for this notebook session [1]: ").strip() or "1"
os.environ["NELLWICK_STUDIO_URL"] = NELLWICK_STUDIO_URL
os.environ["NELLWICK_WORKER_TOKEN"] = NELLWICK_WORKER_TOKEN
os.environ["NELLWICK_MAX_JOBS"] = NELLWICK_MAX_JOBS


In [ ]:
# The public GitHub repository URL is configured after you create/push the repository.
# Do not put your worker token in this cell or commit it to GitHub.
REPOSITORY_URL = "https://github.com/YOUR-GITHUB-USERNAME/nellwick-ai-film-studio.git"
WORKSPACE = "/content/nellwick-ai-film-studio" if "google.colab" in sys.modules else "/kaggle/working/nellwick-ai-film-studio"
if not os.path.isdir(WORKSPACE):
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, WORKSPACE], check=True)
worker_file = os.path.join(WORKSPACE, "workers", "shared", "worker.py")
if not os.path.isfile(worker_file):
    raise FileNotFoundError("workers/shared/worker.py not found. Check REPOSITORY_URL and make sure the project is pushed to GitHub.")
subprocess.run([sys.executable, worker_file, "--max-jobs", NELLWICK_MAX_JOBS], check=True)
